In [1]:
import os
import warnings
from pathlib import Path
from typing import Any

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from model_config import LocalModel, RemoteModel  # noqa: E402
from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = RemoteModel.LLAMA_3_3_70B_INSTRUCT
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),
    base_url=settings.OLLAMA_URL,
    temperature=0.0,
    model=model_str_local,
)

In [5]:
from uuid import uuid4

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

### Load Data

In [ ]:
fp: str = "../../data/chelsea_transfer_news.pdf"
loader = PyPDFLoader(fp)
docs = loader.load()


# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(docs)

console.print(f"Number of chunks: {len(splits)}", style="info")

Number of chunks: 13

### Indexing

In [ ]:
collection_name: str = "cfc_transfer_news"

emb_model = OllamaEmbeddings(
    model=LocalModel.MXBAI_EMBED_LARGE.value,
)
emb = emb_model.embed_documents("Hello world")
emb_size: int = len(emb[0])
console.print(f"Embedding size: {emb_size}", style="info")

client = QdrantClient(url=settings.QDRANT_URL)
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
)

# Vector store
vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=emb_model,
    collection_name=collection_name,
    ids=[str(uuid4()) for _ in range(len(splits))],
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Embedding size: 1024

## Query Transformation

- Query transformations are a set of approaches focused on re-writing and / or modifying questions for retrieval.

<br>

### 1. Multi Query

[![image.png](https://i.postimg.cc/50BsnRfp/image.png)](https://postimg.cc/DJzQz5Jb)

In [8]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template: str = """
<system>
You are an AI language model assistant. 
<role>
Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector 
database.
</role>
<instructions>
By generating multiple perspectives on the user question, your goal is to help the user overcome some of the 
limitations of the distance-based similarity search. Provide these alternative questions separated by newlines. 
<original_question>
{question}
</original_question>
</instructions>
</system>
"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | remote_llm  # local_llm
    | StrOutputParser()
    | (lambda x: x.strip().split("\n"))
)

In [9]:
from IPython.display import Markdown, display

display(Markdown("### Hello"))

### Hello

In [10]:
question: str = "What players are likely to exit Chelsea?"
response = generate_queries.invoke({"question": question})

console.print(question, style="info")
console.print(response)

What players are likely to exit Chelsea?

[
    'Which Chelsea players are expected to leave the club?',
    ' ',
    'Who might be departing Chelsea in the upcoming transfer window?',
    ' ',
    'What Chelsea footballers are rumored to be leaving Stamford Bridge?',
    ' ',
    'Which players are likely to be transferred out of Chelsea?',
    ' ',
    'Who are the Chelsea players that could potentially exit the team in the near future?'
]

In [11]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list[Any]]) -> list[Any]:
    """
    Get the unique union of lists of documents.

    Parameters
    ----------
    documents : list[list[Any]]
        A list of lists containing documents.

    Returns
    -------
    list[Any]
        A list containing unique documents.
    """
    # Flatten list of lists and convert each Document to a string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    unique_docs = list(set(flattened_docs))

    return [loads(doc) for doc in unique_docs]


# Retrieve
question: str = "What players are likely to exit Chelsea?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
retr_docs = retrieval_chain.invoke({"question": question})
len(retr_docs)

/var/folders/vv/g_5scsqs6fj18dr1q_bww19r0000gn/T/ipykernel_83949/1780963150.py:22: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


6

In [12]:
console.print(retr_docs)

[
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 0,
            'page_label': '1',
            '_id': 'e9348421-bcf3-4b0d-ae90-f56af8222741',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates and 
latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: Chelsea 
2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the Sky 
Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naround the deal.\xa0\nKeep scrolling!\xa0\nUPDATE\nF o o t b al l
\n News Watch Scores & FixturesTables Transfers More\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: 
Live updates and latest on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 1/17'
    ),
    Document(
        metadata={
            'producer': 'Skia/PDF m138',
            'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
            'creationdate': '2025-07-25T18:28:43+00:00',
            'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, loans 
and contracts | Football News | Sky Sports',
            'moddate': '2025-07-25T18:28:43+00:00',
            'source': '../../data/chelsea_transfer_news.pdf',
            'total_pages': 12,
            'page': 8,
            'page_label': '9',
            '_id': '58625bd4-6f1f-4c49-ab8b-2bace30503cc',
            '_collection_name': 'cfc_transfer_news'
        },
        page_content="Ajax defender Hato - who spent a lot of last season playing as a left-back - is\none of a 
number of centre-backs Chelsea have been looking at.\n23 Jul\n13:31\nThe players who are likely to leave Chelsea 
this\nsummer...\nChelsea have been linked with a number of incomings - but any further\nsignings this summer will 
be dependent on players leaving, too.\xa0\nHere are the players likely to exit Stamford Bridge in this 
window...\nJoao Felix - The\xa0Portuguese playmaker was not been included in Chelsea's\nClub World Cup squad after 
returning from his loan at AC Milan.\nRaheem Sterling - The winger was also omitted from the Chelsea Club 
World\nCup squad after his Arsenal loan.\xa0\nBen Chilwell - England left-back Chilwell spent the second half of 
last season\non loan at Crystal Palace.\nRenato Veiga\xa0- Veiga joined Juventus on loan in January for the 
remainder of\nlast season.\xa0\nAxel Disasi -\xa0The defender was not included in the Club World Cup squad 
after\nreturning from a loan with Aston Villa.\nCarney Chukwuemeka -\xa0RB Leipzig are pursuing a move for Chelsea 
midfielder\nCarney Chukwuemeka, according to Sky in Germany. Chukwuemeka spent the\nsecond half of last season on 
loan at Borussia Dortmund.\xa0\nChristopher Nkunku -\xa0Inter Milan are 

In [13]:
from operator import itemgetter

# RAG Prompt
rag_template: str = """
<instructions>
Answer the following question based on this context:
<context>{context}</context>
<question>{question}</question
</instructions
"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)
final_rag_chain = (
    {
        "context": retrieval_chain,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | remote_llm  # local_llm
    | StrOutputParser()
)

In [14]:
response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\nResponse:")
display(Markdown(response))

Question: What players are likely to exit Chelsea?
Response:


According to the text, the players who are likely to exit Chelsea this summer are:

1. Joao Felix - The Portuguese playmaker was not included in Chelsea's Club World Cup squad after returning from his loan at AC Milan.
2. Raheem Sterling - The winger was also omitted from the Chelsea Club World Cup squad after his Arsenal loan.
3. Ben Chilwell - England left-back Chilwell spent the second half of last season on loan at Crystal Palace.
4. Renato Veiga - Veiga joined Juventus on loan in January for the remainder of last season.
5. Axel Disasi - The defender was not included in the Club World Cup squad after returning from a loan with Aston Villa.
6. Carney Chukwuemeka - RB Leipzig are pursuing a move for Chelsea midfielder Carney Chukwuemeka, according to Sky in Germany. Chukwuemeka spent the second half of last season on loan at Borussia Dortmund.
7. Christopher Nkunku - Inter Milan are interested in Nkunku, according to Sky in Italy. Nkunku played for Chelsea at the Club World Cup and wanted to join Bayern Munich in January, but a deal could not be agreed.

In [15]:
question: str = "What players are Chelsea FC interested in signing?"
response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\n\nResponse:")
display(Markdown(response))

Question: What players are Chelsea FC interested in signing?

Response:


Based on the provided context, Chelsea FC is interested in signing the following players:

1. Xavi Simons - a forward from RB Leipzig. Chelsea has held talks with RB Leipzig over signing Simons.
2. Jorrel Hato - a centre-back from Ajax. Chelsea is in advanced talks with Ajax over a deal for Hato, with the club ready to pay £44m (€50m), but Ajax is holding out for around £52m (€60m).

These are the two players mentioned in the context as being of interest to Chelsea FC.

In [16]:
question: str = "What players are Chelsea FC interested in signing? Who is Sterling?"
response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\n\nResponse:")
display(Markdown(response))

Question: What players are Chelsea FC interested in signing? Who is Sterling?

Response:


Based on the provided context, Chelsea FC is interested in signing the following players:

1. Xavi Simons - a forward from RB Leipzig. Chelsea has held talks with RB Leipzig over signing Simons.
2. Jorrel Hato - a centre-back from Ajax. Chelsea is in talks to sign Hato, who is one of several centre-backs they have been looking at.

As for Raheem Sterling, he is a 30-year-old winger who currently plays for Chelsea FC. However, he spent last season on loan at Arsenal and is reportedly surplus to requirements at Chelsea. Sterling has two years remaining on his contract at Stamford Bridge, and Chelsea prefers to sell players who are not considered in the manager's plans for the next summer. Fulham has expressed interest in signing Sterling, along with another Chelsea player, Kiernan Dewsbury-Hall.

### 2. RAG FUSION

<br>

[![image.png](https://i.postimg.cc/MH50McSm/image.png)](https://postimg.cc/8fFfxzs7)

In [ ]:
# RAG-Fusion: Related
template = """
<system>
<role>
You are a helpful assistant that generates multiple search queries based on a single input query.
</role>

<instructions>
Generate a max of 4 search queries related to the question
<question>{question}</question>
</instructions>

<outputs>
Output:
</outputs>

</system>
"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)
generate_queries = prompt_rag_fusion | remote_llm | StrOutputParser() | (lambda x: x.strip().split("\n"))

In [ ]:
def reciprocal_rank_fusion(results: list[list], k: int = 60) -> list[tuple[Any, Any]]:
    """
    Apply Reciprocal Rank Fusion (RRF) to combine multiple ranked document lists.

    RRF is a method for combining multiple ranked lists of documents into a single
    ranked list. It assigns scores to documents based on their ranks across all
    input lists using the formula: 1 / (rank + k), where k is a constant that
    controls the influence of lower-ranked documents.

    Parameters
    ----------
    results : list[list]
        A list of lists, where each inner list contains ranked documents
        from different retrieval methods or queries.
    k : int, default=60
        The RRF constant parameter. Higher values reduce the difference
        between ranks, making the fusion less sensitive to rank differences.
        Typical values range from 10 to 100.

    Returns
    -------
    list[tuple[Any, Any]]
        A list of tuples where each tuple contains:
        - First element: The document object (deserialized from JSON)
        - Second element: The fused score (float)
        Documents are sorted by fused score in descending order.

    Notes
    -----
    - Documents are serialized to JSON strings for deduplication
    - The RRF formula ensures that documents appearing in multiple lists
      get higher combined scores
    - Documents with better ranks (lower rank numbers) receive higher scores

    Examples
    --------
    >>> doc_lists = [[doc1, doc2], [doc2, doc3], [doc1, doc3]]
    >>> fused = reciprocal_rank_fusion(doc_lists, k=60)
    >>> # Returns documents ranked by their combined RRF scores
    """
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    return [(loads(doc), score) for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)]

In [19]:
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
question: str = "Who is Xavi Simmons?"

docs = retrieval_chain_rag_fusion.invoke({"question": question})
print(len(docs))
print(f"Question: {question}\n\n")
console.print(docs)

6
Question: Who is Xavi Simmons?




[
    (
        Document(
            metadata={
                'producer': 'Skia/PDF m138',
                'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
                'creationdate': '2025-07-25T18:28:43+00:00',
                'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, 
loans and contracts | Football News | Sky Sports',
                'moddate': '2025-07-25T18:28:43+00:00',
                'source': '../../data/chelsea_transfer_news.pdf',
                'total_pages': 12,
                'page': 5,
                'page_label': '6',
                '_id': 'ed2de579-cc84-4d25-9b9a-e9edaa8b06d1',
                '_collection_name': 'cfc_transfer_news'
            },
            page_content='24 Jul\n16:05\nWhat is the latest on interest from Chelsea for\nSimons?\nSky Germany’s RB
Leipzig reporter Philipp Hinze:\xa0\n"Chelsea and Leipzig are in direct contact. Talks are ongoing. 
Negotiations\nwith the player’s side are very constructive.\xa0\n"However, Leipzig’s demands are clear: they are 
still asking for a total package\nof around €70 million, including add-ons. They do not want to let Xavi go 
for\nless. They have a clear stance.\xa0\n"However, they are also aware that Xavi wants to leave — this is 
known\ninternally and has been clearly discussed."\n24 Jul\n16:04\nWhat is Simons\' current situation with 
Leipzig\'s pre-\nseason?\nSky Germany’s RB Leipzig reporter Philipp Hinze:\xa0\n"It was clearly agreed that if 
there is no agreement on a transfer with an\ninterested club before the training camp, then Xavi will travel with 
the team to\nthe camp.\xa0\n"A normal situation — Xavi is still a Leipzig player and is training regularly 
with\nthe team without any restrictions. He is behaving positively and is in good\nspirits. There are no signs of a
bad mood. Xavi is showing good form in\ntraining."\n24 Jul\n15:55\nIsak not a target for Chelsea\nLatest from Sky 
Sports News\' Kaveh Solhekol:\n25/07/2025, 19:28 Chelsea transfer news, rumours and gossip: Live updates and latest
on deals, signings, loans and contracts | Football News | Sky 
Sports\nhttps://www.skysports.com/football/live-blog/11668/13025497/chelsea-transfer-news-rumours-and-gossip-live-u
pdates-and-latest-on-deals-signings-loans-and-… 6/17'
        ),
        0.08172043010752687
    ),
    (
        Document(
            metadata={
                'producer': 'Skia/PDF m138',
                'creator': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) 
Chrome/138.0.0.0 Safari/537.36',
                'creationdate': '2025-07-25T18:28:43+00:00',
                'title': 'Chelsea transfer news, rumours and gossip: Live updates and latest on deals, signings, 
loans and contracts | Football News | Sky Sports',
                'moddate': '2025-07-25T18:28:43+00:00',
                'source': '../../data/chelsea_transfer_news.pdf',
                'total_pages': 12,
                'page': 0,
                'page_label': '1',
                '_id': 'e9348421-bcf3-4b0d-ae90-f56af8222741',
                '_collection_name': 'cfc_transfer_news'
            },
            page_content='Friday 25 July 2025 14:39, UK\nChelsea transfer news, rumours and\ngossip: Live updates 
and latest on\ndeals, signings, loans and contracts\nLatest Chelsea news\xa0\nSort by: Latest Oldest\nIn full: 
Chelsea 2025/26 Premier League fixtures\xa0\nTransfer Centre LIVE! Deals, rumours, news on your phone\nDownload the
Sky Sports app for Chelsea transfers, analysis and FREE highlights from\nEVERY Premier League game\xa0\xa0\xa0View 
post\n24 Jul\n16:22 Simons to Chelsea? The key questions answered...\nChelsea have held talks over signing RB 
Leipzig forward Xavi Simons - but how\nmuch will they have to pay and who else want him?\xa0\nSky Germany’s Leipzig
reporter Philipp Hinze answers the key questions\naroun

In [20]:
# RAG Prompt
rag_template: str = """
<instructions>
Answer the following question based on this context:
<context>{context}</context>
<question>{question}</question
</instructions
"""
rag_prompt = ChatPromptTemplate.from_template(rag_template)
final_rag_chain = (
    {
        "context": retrieval_chain_rag_fusion,
        "question": itemgetter("question"),
    }
    | rag_prompt
    | remote_llm  # local_llm
    | StrOutputParser()
)

In [21]:
question: str = "Who is Xavi Simmons?"

response = final_rag_chain.invoke({"question": question})
print(f"Question: {question}\n\nResponse:")
display(Markdown(response))

Question: Who is Xavi Simmons?

Response:


Xavi Simons is a football player who currently plays for RB Leipzig. According to the text, he is being considered for a transfer to Chelsea, with Leipzig demanding a total package of around €70 million, including add-ons. Simons has performed well in the Bundesliga, despite a serious injury that held him back for several months. He sees himself as a leader at Leipzig and is looking to take the next step in his development, with Leipzig and the Bundesliga being a major opportunity for his growth.

In [ ]:
# uvr -m streamlit run frontend/main.py


from operator import itemgetter
from typing import Any
from uuid import uuid4

import streamlit as st
from langchain.prompts import ChatPromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSerializable
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from model_config import LocalModel, RemoteModel
from settings import refresh_settings

settings = refresh_settings()

model_str_remote: str = RemoteModel.LLAMA_3_3_70B_INSTRUCT
model_str_local: str = LocalModel.MISTRAL_7B_INSTRUCT_V0_3_Q4_0

# Deterministic responses
remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,  # type: ignore
    temperature=0.0,
    model=model_str_remote,  # type: ignore
)

local_llm = ChatOpenAI(
    api_key=settings.OLLAMA_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OLLAMA_URL,  # type: ignore
    temperature=0.0,
    model=model_str_local,  # type: ignore
)

llm = remote_llm


def get_document_splits(filepath: str) -> list[Any]:
    """Load a PDF file and split it into smaller chunks."""
    loader = PyPDFLoader(filepath)
    docs = loader.load()

    # Split the documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=500, chunk_overlap=100)
    return text_splitter.split_documents(docs)


def get_vector_store(splits: list[Any], collection_name: str) -> VectorStoreRetriever:
    """Create a vector store retriever from document splits."""
    emb_model = OllamaEmbeddings(
        model=LocalModel.MXBAI_EMBED_LARGE.value,
    )
    emb = emb_model.embed_documents(["Hello world"])
    emb_size: int = len(emb[0])

    client = QdrantClient(url=settings.QDRANT_URL)
    client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=emb_size, distance=Distance.COSINE),
    )

    # Vector store
    vector_store = QdrantVectorStore.from_documents(
        documents=splits,
        embedding=emb_model,
        collection_name=collection_name,
        ids=[str(uuid4()) for _ in range(len(splits))],
    )
    return vector_store.as_retriever(search_kwargs={"k": 3})


def get_rag_fusion_generator(llm: ChatOpenAI) -> RunnableSerializable[dict, Any]:
    """Generate RAG fusion queries."""
    template = """
    <system>
    <role>
    You are a helpful assistant that generates multiple search queries based on a single input query.
    </role>

    <instructions>
    Generate a max of 4 search queries related to the question
    <question>{question}</question>
    </instructions>

    <outputs>
    Output:
    </outputs>

    </system>
    """
    prompt_rag_fusion = ChatPromptTemplate.from_template(template)
    return prompt_rag_fusion | llm | StrOutputParser() | (lambda x: x.strip().split("\n"))


def reciprocal_rank_fusion(results: list[list], k: int = 60) -> list[tuple[Any, Any]]:
    """
    Apply Reciprocal Rank Fusion (RRF) to combine multiple ranked document lists.
    """
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    return [(loads(doc), score) for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)]


def get_final_rag_pipeline(retrieval_chain: Any, llm: ChatOpenAI) -> RunnableSerializable[dict, Any]:
    """Generate the final RAG pipeline."""
    # RAG Prompt
    rag_template: str = """
    <instructions>
    Answer the following question based on this context:
    <context>{context}</context>
    <question>{question}</question>
    </instructions>
    """
    rag_prompt = ChatPromptTemplate.from_template(rag_template)
    return (
        {
            "context": retrieval_chain,
            "question": itemgetter("question"),
        }
        | rag_prompt
        | llm
        # | StrOutputParser()
    )


# Streamlit UI
st.title("Simple RAG System")

# Initialize session state
if "rag_chain" not in st.session_state:
    st.session_state.rag_chain = None
if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    uploaded_file = st.file_uploader("Upload a PDF file", type=["pdf"])

    if uploaded_file:
        # Save uploaded file temporarily
        with open(f"temp_{uploaded_file.name}", "wb") as f:
            f.write(uploaded_file.read())

        # Process the document
        with st.spinner("Processing document..."):
            splits = get_document_splits(f"temp_{uploaded_file.name}")
            generate_queries = get_rag_fusion_generator(llm=llm)
            retriever = get_vector_store(splits, "demo_collection")
            retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
            st.session_state.rag_chain = get_final_rag_pipeline(retrieval_chain_rag_fusion, llm=llm)

        st.success(f"Successfully processed {len(splits)} document chunks!")

# Display chat messages from history on app rerun
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept user input
if prompt := st.chat_input("What's on your mind?"):
    if st.session_state.rag_chain is None:
        st.warning("Please upload a PDF file first!")
    else:
        # Add user message to chat history
        st.session_state.messages.append({"role": "user", "content": prompt})
        # Display user message in chat message container
        with st.chat_message("user"):
            st.markdown(prompt)

        # Display assistant response in chat message container
        with st.chat_message("assistant"):

            def response_generator():
                for chunk in st.session_state.rag_chain.stream({"question": prompt}):
                    if hasattr(chunk, "content"):
                        yield chunk.content
                    elif isinstance(chunk, str):
                        yield chunk
                    else:
                        yield str(chunk)

            response = st.write_stream(response_generator())

        # Add assistant response to chat history
        st.session_state.messages.append({"role": "assistant", "content": response})


In [ ]:
# how many players are chelsea fc interested in signing?
# Any news about Sterling?
# Who is Xavi and what club does he play for?
# What are the names of the reporters that wrote the blog or article about chelsea transfers?